# LLM Reliability Analytics - EDA Notebook

This notebook provides a simple exploratory analysis workflow for `test_results` data.
It is structured for a first master's defense demo: clear metrics, clear plots, minimal complexity.

## 1) Imports

In [ ]:
import os
from pathlib import Path

import duckdb
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('ggplot')
%matplotlib inline

## 2) Configure Data Source

Set `USE_DUCKDB = True` to read from DuckDB, or `False` to read from CSV.

In [ ]:
USE_DUCKDB = True

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
duckdb_env_path = os.getenv('LLM_RELIABILITY_DB_PATH')
DUCKDB_PATH = Path(duckdb_env_path) if duckdb_env_path else (PROJECT_ROOT / 'data' / 'reliability.duckdb')
CSV_PATH = PROJECT_ROOT / 'data' / 'processed' / 'test_results.csv'

print('Project root:', PROJECT_ROOT)
print('DuckDB path:', DUCKDB_PATH)
print('CSV path:', CSV_PATH)

## 3) Load `test_results` from DuckDB or CSV

In [ ]:
if USE_DUCKDB:
    if not DUCKDB_PATH.exists():
        raise FileNotFoundError(f'DuckDB file not found: {DUCKDB_PATH}')

    conn = duckdb.connect(str(DUCKDB_PATH))
    try:
        df = conn.execute('SELECT * FROM test_results').fetchdf()
    finally:
        conn.close()
else:
    if not CSV_PATH.exists():
        raise FileNotFoundError(f'CSV file not found: {CSV_PATH}')
    df = pd.read_csv(CSV_PATH)

print('Loaded rows:', len(df))
df.head()

## 4) Dataset Shape and Schema

In [ ]:
print('Shape:', df.shape)
print('Columns:', list(df.columns))
print('\nSchema:')
df.info()

## 5) Basic Data Preparation

In [ ]:
if df.empty:
    raise ValueError('No rows in test_results. Run a batch first, then re-open this notebook.')

df = df.copy()

if 'is_correct' in df.columns:
    df['is_correct'] = df['is_correct'].astype(bool)
else:
    raise KeyError("Column 'is_correct' is required for analysis.")

if 'latency_ms' in df.columns:
    df['latency_ms'] = pd.to_numeric(df['latency_ms'], errors='coerce')
else:
    raise KeyError("Column 'latency_ms' is required for analysis.")

df['category'] = df.get('category', 'unknown').fillna('unknown').astype(str)
df['error_type'] = df.get('error_type').fillna('none').astype(str)

df[['is_correct', 'latency_ms', 'category', 'error_type']].head()

## 6) Pass/Fail Counts and Overall Accuracy

In [ ]:
total = len(df)
passed = int(df['is_correct'].sum())
failed = total - passed
accuracy = passed / total if total > 0 else 0.0

print(f'Total: {total}')
print(f'Passed: {passed}')
print(f'Failed: {failed}')
print(f'Accuracy: {accuracy:.3f}')

pd.DataFrame({'metric': ['total', 'passed', 'failed', 'accuracy'], 'value': [total, passed, failed, accuracy]})

## 7) Accuracy by Category

In [ ]:
category_accuracy = (
    df.groupby('category', as_index=False)['is_correct']
      .mean()
      .rename(columns={'is_correct': 'accuracy'})
      .sort_values('accuracy', ascending=False)
)
category_accuracy

## 8) Latency Distribution

In [ ]:
latency_stats = df['latency_ms'].describe(percentiles=[0.5, 0.9, 0.95]).to_frame(name='latency_ms')
latency_stats

## 9) Error Type Frequencies

In [ ]:
error_freq = df['error_type'].value_counts(dropna=False).rename_axis('error_type').reset_index(name='count')
error_freq

## 10) Simple Matplotlib Charts

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Pass/Fail bar
axes[0, 0].bar(['passed', 'failed'], [passed, failed], color=['#2ca02c', '#d62728'])
axes[0, 0].set_title('Pass vs Fail')
axes[0, 0].set_ylabel('Count')

# Category accuracy bar
axes[0, 1].bar(category_accuracy['category'], category_accuracy['accuracy'], color='#1f77b4')
axes[0, 1].set_title('Accuracy by Category')
axes[0, 1].set_ylabel('Accuracy')
axes[0, 1].set_ylim(0, 1)
axes[0, 1].tick_params(axis='x', rotation=45)

# Latency histogram
axes[1, 0].hist(df['latency_ms'].dropna(), bins=15, color='#ff7f0e', edgecolor='black')
axes[1, 0].set_title('Latency Distribution (ms)')
axes[1, 0].set_xlabel('Latency (ms)')
axes[1, 0].set_ylabel('Frequency')

# Error frequency bar
axes[1, 1].bar(error_freq['error_type'], error_freq['count'], color='#9467bd')
axes[1, 1].set_title('Error Type Frequency')
axes[1, 1].set_ylabel('Count')
axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 11) Demo Summary

For presentation, summarize:
- dataset size and schema
- overall pass/fail and accuracy
- strongest and weakest categories
- latency profile (median/p95)
- dominant error types